In [ ]:
# parameters
input_image = None


In [9]:
# -*- coding: utf-8 -*-
"""
Split combined OCT+mask images (left=OCT, right=colored mask) into the project's
processed layout:
  processed/
    images/<rel>.png
    masks/<rel>.png   # 8-bit IDs 0..5
    txt/fold_1/train.txt   # lines: relative path WITHOUT extension (e.g., 12/12_3)
    txt/fold_1/val.txt

Edit SRC_ROOT / OUT_ROOT / VAL_RATIO below, then run:
    python prepare_amd_sd_processed.py
"""

import os, random
from pathlib import Path
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

# ----------------- CONFIG (EDIT ME) -----------------
SRC_ROOT = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\AMD-SD\images"  # combined images
OUT_ROOT = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed"      # target processed dir
VAL_RATIO = 0.20   # 20% for validation
SEED = 2025
# ----------------------------------------------------

EXS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
PALETTE = {0:(0,0,0), 1:(255,0,0), 2:(0,0,255), 3:(0,255,0), 4:(255,255,0), 5:(255,0,255)}

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def scan_images(root: Path):
    files = []
    for p in root.rglob("*"):
        if p.suffix.lower() in EXS and p.is_file():
            files.append(p)
    files.sort()
    return files

def rgb_mask_to_ids_hsv(rgb: np.ndarray, sat_thr: int = 40, val_thr: int = 40) -> np.ndarray:
    """Tolerant mapping RGB->class IDs using HSV thresholds.
       ID mapping (consistent with earlier AMD-SD code):
         0 = background
         1 = red       (SRF)
         2 = blue      (PED)
         3 = green     (IRF)
         4 = yellow    (SHRM)
         5 = magenta   (EZ defect)
    """
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    H,S,V = hsv[...,0], hsv[...,1], hsv[...,2]
    idm = np.zeros(H.shape, np.uint8)
    bg = (S < sat_thr) | (V < val_thr)
    red     = ((H <= 10) | (H >= 170)) & (~bg)   # 1 SRF
    yellow  = (H >= 20) & (H <= 40)   & (~bg)    # 4 SHRM
    green   = (H >= 45) & (H <= 85)   & (~bg)    # 3 IRF
    blue    = (H >= 100)& (H <= 130)  & (~bg)    # 2 PED
    magenta = (H >= 140)& (H <= 170)  & (~bg)    # 5 EZ defect
    idm[red]     = 1
    idm[blue]    = 2
    idm[green]   = 3
    idm[yellow]  = 4
    idm[magenta] = 5
    return idm

def main():
    src_root = Path(SRC_ROOT)
    out_root = Path(OUT_ROOT)
    assert src_root.is_dir(), f"Source dir not found: {src_root}"
    ensure_dir(out_root)

    img_out = ensure_dir(out_root / "images")
    msk_out = ensure_dir(out_root / "masks")
    txt_dir = ensure_dir(out_root / "txt" / "fold_1")

    files = scan_images(src_root)
    if not files:
        raise SystemExit(f"No images found under: {src_root}")

    kept_rel_noext = []
    cls_hist = np.zeros(6, np.int64)

    for p in tqdm(files, desc="Split & convert"):
        im = Image.open(p).convert("RGB")
        W, H = im.size
        mid = W // 2

        arr = np.array(im)
        left  = arr[:, :mid, :]    # OCT
        right = arr[:, mid:, :]    # mask (colored)

        # OCT → grayscale 8-bit
        left_gray = cv2.cvtColor(left, cv2.COLOR_RGB2GRAY)

        # Mask → IDs 0..5
        idmask = rgb_mask_to_ids_hsv(right)

        # derive relative stem based on source root (without extension)
        rel = p.relative_to(src_root).with_suffix("")  # keep folders, drop ext
        # save to processed/images and processed/masks with same relative structure
        img_path = img_out / (rel.as_posix() + ".png")
        msk_path = msk_out / (rel.as_posix() + ".png")
        ensure_dir(img_path.parent)
        ensure_dir(msk_path.parent)

        # write files
        Image.fromarray(left_gray, mode="L").save(img_path)
        Image.fromarray(idmask,   mode="L").save(msk_path)

        kept_rel_noext.append(rel.as_posix())
        # update histogram
        h, _ = np.histogram(idmask, bins=[0,1,2,3,4,5,6])
        cls_hist[:6] += h

    # Split train/val
    random.seed(SEED)
    random.shuffle(kept_rel_noext)
    n_val = int(len(kept_rel_noext) * VAL_RATIO)
    val_ids  = kept_rel_noext[:n_val]
    train_ids= kept_rel_noext[n_val:]

    # write lists: lines are relative path WITHOUT extension
    with open(txt_dir / "train.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(train_ids))
    with open(txt_dir / "val.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(val_ids))

    print("\nDone.")
    print(f"Processed images saved to: {img_out}")
    print(f"Processed masks  saved to: {msk_out}")
    print(f"Splits written to:          {txt_dir}")
    print(f"Counts → train: {len(train_ids)} | val: {len(val_ids)} | total: {len(kept_rel_noext)}")
    print("Class pixel histogram (0..5):", cls_hist.tolist())

if __name__ == "__main__":
    main()


Split & convert: 100%|█████████████████████████████████████████████████████████████| 3049/3049 [01:23<00:00, 36.53it/s]


Done.
Processed images saved to: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed\images
Processed masks  saved to: F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed\masks
Splits written to:          F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed\txt\fold_1
Counts → train: 2440 | val: 609 | total: 3049
Class pixel histogram (0..5): [633900998, 6429538, 7571091, 1331359, 6131134, 5049280]


In [10]:
# ============================ Cell 0 · 路径 & 环境 ============================
import sys, os, platform, random
from pathlib import Path

# 1) 修改为你的项目根目录（包含 models/、dataloaders/ 等）
PROJECT_ROOT = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main"  # ← 改成你的
assert os.path.isdir(PROJECT_ROOT), f"项目根目录不存在：{PROJECT_ROOT}"

# 2) 数据根目录（processed 目录，通常含 images/、masks/、txt/fold_1/）
DATA_ROOT = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed"   # ← 改成你的
assert os.path.isdir(DATA_ROOT), f"数据根目录不存在：{DATA_ROOT}"

# 3) 输出目录（按你的 config）
OUT_DIR = r"F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\checkpoints\OCTnext"  # ← 改成你的
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# 4) 放入 sys.path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 5) 常用库
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed=2025):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(2025)
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

PyTorch: 2.5.1 | CUDA: False


In [11]:
# ======================= Cell 1 · utils.* 软补丁（内存注入） =======================
# 目的：完整复现“从原项目导入”，同时避免 utils.* 缺失导致的导入错误
import types, importlib, logging

# 顶层 utils 包
if 'utils' not in sys.modules:
    sys.modules['utils'] = types.ModuleType('utils')

# 1) utils.torchsummary
ts = types.ModuleType('utils.torchsummary')
def summary(*args, **kwargs): return ""
ts.summary = summary
sys.modules['utils.torchsummary'] = ts
sys.modules['utils'].torchsummary = ts

# 2) utils.helpers
helpers = sys.modules.get('utils.helpers') or types.ModuleType('utils.helpers')
def initialize_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

def get_upsampling_weight(in_channels, out_channels, kernel_size):
    """FCN 反卷积双线性初始化（供 models/fcn.py 等使用）"""
    import numpy as _np
    factor = (kernel_size + 1) // 2
    center = factor - 1 if (kernel_size % 2 == 1) else factor - 0.5
    og = _np.ogrid[:kernel_size, :kernel_size]
    filt = (1 - abs(og[0] - center) / factor) * (1 - abs(og[1] - center) / factor)
    w = _np.zeros((in_channels, out_channels, kernel_size, kernel_size), dtype=_np.float32)
    for i in range(min(in_channels, out_channels)):
        w[i, i, ...] = filt
    return torch.from_numpy(w)

def set_trainable(module, requires_grad=True):
    for p in module.parameters(): p.requires_grad = requires_grad

def dir_exists(p): os.makedirs(p, exist_ok=True)

helpers.initialize_weights = initialize_weights
helpers.weights_init = initialize_weights
helpers.get_upsampling_weight = get_upsampling_weight
helpers.set_trainable = set_trainable
helpers.dir_exists = dir_exists
sys.modules['utils.helpers'] = helpers
sys.modules['utils'].helpers = helpers

# 3) utils.lr_scheduler.Poly
lrmod = types.ModuleType('utils.lr_scheduler')
class Poly(object):
    def __init__(self, optimizer, epochs, iters_per_epoch, power=0.9):
        self.optimizer = optimizer
        self.T = max(1, epochs * max(1, iters_per_epoch))
        self.t = 0
        self.base_lrs = [g.get('lr', 1e-3) for g in optimizer.param_groups]
        self.power = power
    def step(self):
        self.t += 1
        ratio = min(1.0, self.t / self.T)
        for base_lr, g in zip(self.base_lrs, self.optimizer.param_groups):
            g['lr'] = base_lr * ((1 - ratio) ** self.power)
lrmod.Poly = Poly
sys.modules['utils.lr_scheduler'] = lrmod
sys.modules['utils'].lr_scheduler = lrmod

# 4) utils.sync_batchnorm
syncbn = types.ModuleType('utils.sync_batchnorm')
def convert_model(m): return m
class DataParallelWithCallback(torch.nn.DataParallel): pass
syncbn.convert_model = convert_model
syncbn.DataParallelWithCallback = DataParallelWithCallback
sys.modules['utils.sync_batchnorm'] = syncbn
sys.modules['utils'].sync_batchnorm = syncbn

# 5) utils.logger
ulog = types.ModuleType('utils.logger')
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(name)s - %(levelname)s - %(message)s')
ulog.logging = logging
sys.modules['utils.logger'] = ulog
sys.modules['utils'].logger = ulog

# 6) utils.palette（备用）
pal = types.ModuleType('utils.palette')
def get_voc_palette(num_classes):
    def bitget(byteval, idx): return (byteval & (1 << idx)) != 0
    palette = []
    for j in range(num_classes):
        lab = j; r = g = b = 0; i = 0
        while lab:
            r |= (bitget(lab, 0) << (7 - i))
            g |= (bitget(lab, 1) << (7 - i))
            b |= (bitget(lab, 2) << (7 - i))
            i += 1; lab >>= 3
        palette.extend([r, g, b])
    return palette
pal.get_voc_palette = get_voc_palette
sys.modules['utils.palette'] = pal
sys.modules['utils'].palette = pal

print("utils.* 软补丁：OK")

utils.* 软补丁：OK


In [12]:
# ================== Cell 2 · 读取项目 config（以你提供的 JSON 为准） ==================
CONFIG = {
  "arch": {
    "args": {
      "freeze_backbone": False,
      "freeze_bn": False
    },
    "smp": {
      "decoder_name": "Unet",
      "encoder_name": "resnet50"
    },
    "type": "unet"
  },
  "class_num": 6,
  "ignore_index": 255,
  "loss": "CE_DiceLoss",
  "lr_scheduler": { "args": {}, "type": "Poly" },
  "n_gpu": 1,
  "name": "",
  "optimizer": { "args": { "lr": 1e-4 }, "differential_lr": False, "type": "AdamW" },
  "train_loader": {
    "args": {
      "augment": True,
      "base_size": False,
      "batch_size": 8,
      "clahe": False,
      "crop_size": 512,
      "data_dir": DATA_ROOT,
      "flip": True,
      "num_workers": 20,
      "randomblur": False,
      "randombrightness": False,
      "randomcontrast": False,
      "rotate": True,
      "scale": False,
      "shuffle": True,
      "split": "train",
      "square_resize": 512
    },
    "type": "OCT"
  },
  "trainer": {
    "early_stop": 50,
    "epochs": 6,
    "log_dir": "checkpoints/runs8",
    "log_per_iter": 20,
    "monitor": "max average_dice_score",
    "save_dir": OUT_DIR,
    "save_period": 100,
    "tensorboard": True,
    "val": True,
    "val_per_epochs": 1
  },
  "use_smp": False,
  "use_synch_bn": False,
  "val_loader": {
    "args": {
      "batch_size": 8,
      "clahe": False,
      "crop_size": 512,
      "data_dir": DATA_ROOT,
      "num_workers": 16,
      "split": "val",
      "square_resize": 512,
      "val": True
    },
    "type": "OCT"
  }
}
print("CONFIG: epochs =", CONFIG["trainer"]["epochs"], "| class_num =", CONFIG["class_num"])

CONFIG: epochs = 6 | class_num = 6


In [13]:
# =========== Cell 3 · 修复 oct.py 的 root 硬编码 + 生成 {train,val}.txt（无扩展名） ===========
import os, sys, importlib, random
from pathlib import Path

# 1) 先 reload 再 monkey-patch _set_files：不再覆盖 root，按 <root>/txt/fold_1/{split}.txt
if 'dataloaders.oct' in sys.modules:
    importlib.reload(sys.modules['dataloaders.oct'])
import dataloaders.oct as oct_mod

def _patched_set_files(self):
    self.image_dir = os.path.join(self.root, "images")
    self.label_dir = os.path.join(self.root, "masks")
    file_list = os.path.join(self.root, "txt", "fold_1", self.split + ".txt")
    with open(file_list, "r", encoding="utf-8") as f:
        # 每行：相对 images 的路径（不带扩展名）
        self.files = [ln.strip() for ln in f if ln.strip()]

# 在 reload 之后打补丁（关键）
oct_mod.OCTDataset._set_files = _patched_set_files

# 2) 在 DATA_ROOT 下生成 splits（只做 train/val）
VAL_RATIO = 0.20
OCT_ROOT = Path(CONFIG["train_loader"]["args"]["data_dir"])
img_dir  = OCT_ROOT / "images"
mask_dir = OCT_ROOT / "masks"
txt_dir  = OCT_ROOT / "txt" / "fold_1"
txt_dir.mkdir(parents=True, exist_ok=True)

assert img_dir.is_dir(), f"找不到图像目录：{img_dir}"
assert mask_dir.is_dir(), f"找不到掩膜目录：{mask_dir}"

# 仅收集 .png 且图像/掩膜都存在；条目为“相对 images 的路径（无扩展名）”
all_ids = []
for p in sorted(img_dir.rglob("*.png")):
    rel = p.relative_to(img_dir)        # e.g. patient001/scanA/000123.png
    rel_no_ext = rel.with_suffix("")    # e.g. patient001/scanA/000123
    if (mask_dir / (rel_no_ext.as_posix() + ".png")).exists():
        all_ids.append(rel_no_ext.as_posix())

print(f"可用 PNG 成对样本数：{len(all_ids)}")

random.seed(2025)
random.shuffle(all_ids)
n_val = int(len(all_ids) * VAL_RATIO)
val_ids  = all_ids[:n_val]
train_ids= all_ids[n_val:]

def _write_list(path, items):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for it in items: f.write(it + "\n")

_write_list(txt_dir / "train.txt", train_ids)
_write_list(txt_dir / "val.txt",   val_ids)

print(f"写入列表：train={len(train_ids)} | val={len(val_ids)} → {txt_dir}")
print("train 示例：", train_ids[:3])
print("val   示例：", val_ids[:3])

可用 PNG 成对样本数：3049
写入列表：train=2440 | val=609 → F:\Basic-Seg-Experiment-main\Basic-Seg-Experiment-main\AMD-SD\processed\txt\fold_1
train 示例： ['82/82_2', '120/120_1', '121/121_22']
val   示例： ['141/141_9', '82/82_1', '12/12_3']


In [14]:
# ================== Cell 4 · 导入原项目的数据集 & 模型 ==================
# 数据集：使用 dataloaders.oct.OCT（原项目实现）
from dataloaders.oct import OCT

# 模型：使用原项目 UNet（models/__init__.py 会引入其他模型，已在 Cell1 软补丁兼容）
from models.unet import UNet, UNetResnet

print("导入成功：", OCT, UNet)

导入成功： <class 'dataloaders.oct.OCT'> <class 'models.unet.UNet'>


In [15]:
# ================== Cell 5 · 构建 Loader（直接使用 OCT；不要再套 DataLoader） ==================
import platform, inspect

def filter_kwargs(fn, cfg: dict):
    import inspect as _inspect
    sig = _inspect.signature(fn)
    return {k: v for k, v in cfg.items() if k in sig.parameters}

train_args = CONFIG["train_loader"]["args"].copy()
val_args   = CONFIG["val_loader"]["args"].copy()

# Windows 下多进程容易卡，安全降级
if platform.system() == "Windows":
    train_args["num_workers"] = 0
    val_args["num_workers"]   = 0

# 保证 split 与根目录正确
train_args["split"]    = "train"
train_args["data_dir"] = str(DATA_ROOT)
val_args["split"]      = "val"
val_args["data_dir"]   = str(DATA_ROOT)

# 仅保留 OCT.__init__ 接受的参数
oct_train_kwargs = filter_kwargs(OCT.__init__, train_args)
oct_val_kwargs   = filter_kwargs(OCT.__init__, val_args)

# 关键：直接实例化 OCT（它本身就是可迭代 loader）
train_loader = OCT(**oct_train_kwargs)
val_loader   = OCT(**oct_val_kwargs)

print(f"[OK] Loader 就绪  train(batches): {len(train_loader)} | val(batches): {len(val_loader)}")
# 如果想看一个 batch 的张量形状，可以取消下面两行的注释：
# xb, yb, *rest = next(iter(train_loader))
# print("sample batch:", xb.shape, yb.shape)


[OK] Loader 就绪  train(batches): 305 | val(batches): 77


In [16]:
xb, yb = next(iter(train_loader))
print("x:", xb.shape, xb.min().item(), xb.max().item())  # 例如 [B,3,512,512]，已标准化
print("y:", yb.shape, yb.min().item(), yb.max().item())

x: torch.Size([8, 3, 512, 512]) -1.908372402191162 2.466442108154297
y: torch.Size([8, 512, 512]) 0 5


In [17]:
# ================== Cell 6 · 构建模型、损失、优化器、PolyLR（自动对齐 in_ch/num_classes） ==================
from utils.lr_scheduler import Poly

DEVICE = "cuda" if (torch.cuda.is_available() and CONFIG["n_gpu"]>=1) else "cpu"

# ---- 动态探测：输入通道数 & 数据集类别数 ----
# 取一个 batch 看看通道
_probe_batch = next(iter(train_loader))
if isinstance(_probe_batch, (list, tuple)) and len(_probe_batch) >= 2:
    xb, yb = _probe_batch[0], _probe_batch[1]
else:
    raise RuntimeError("Dataset 返回格式不含标签 y。")
IN_CH = int(xb.shape[1])
# 从数据集读取类别数（oct.py 里通常是 4）；若没有该属性就沿用 config
DATASET_NUM_CLASSES = int(getattr(getattr(train_loader, "dataset", None), "num_classes",
                                  CONFIG["class_num"]))

# 若与配置不同，进行对齐（避免 one-hot 维度不一致）
if DATASET_NUM_CLASSES != CONFIG["class_num"]:
    print(f"[WARN] 覆盖 class_num: config={CONFIG['class_num']} -> dataset={DATASET_NUM_CLASSES}")
    CONFIG["class_num"] = DATASET_NUM_CLASSES

print(f"[INFO] 输入通道 IN_CH = {IN_CH}，类别数 class_num = {CONFIG['class_num']}")

# ---- 模型：按实测输入通道构建（use_smp = False 用自带 UNet；也可换 UNetResnet）----
if CONFIG.get("use_smp", False):
    import segmentation_models_pytorch as smp
    model = smp.Unet(encoder_name=CONFIG["arch"]["smp"]["encoder_name"],
                     classes=CONFIG["class_num"], in_channels=IN_CH)
else:
    # 原项目 UNet
    model = UNet(in_channels=IN_CH, num_classes=CONFIG["class_num"])

# 冻结策略
if CONFIG["arch"]["args"].get("freeze_backbone", False) and hasattr(model, "backbone"):
    for p in model.backbone.parameters(): p.requires_grad = False
if CONFIG["arch"]["args"].get("freeze_bn", False):
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
            m.eval()
            for p in m.parameters(): p.requires_grad = False

# （可选）SyncBN
if CONFIG.get("use_synch_bn", False):
    from utils.sync_batchnorm import convert_model
    model = convert_model(model)

model = model.to(DEVICE)

# 损失：CE(ignore_index=255) + 前景 Dice
ce = nn.CrossEntropyLoss(ignore_index=CONFIG["ignore_index"])

def dice_loss(logits: torch.Tensor, target: torch.Tensor, classes: int, ignore_index: int) -> torch.Tensor:
    mask_valid = (target != ignore_index)
    target = target.clone()
    target[~mask_valid] = 0
    prob = torch.softmax(logits, dim=1)
    oh = torch.nn.functional.one_hot(target, num_classes=classes).permute(0,3,1,2).float()
    oh[:, :, ~mask_valid] = 0
    eps = 1e-6
    dice_sum, k = 0.0, 0
    for c in range(1, classes):
        inter = (prob[:, c] * oh[:, c]).sum()
        denom = prob[:, c].sum() + oh[:, c].sum() + eps
        dice_sum += (1 - 2*inter/denom); k += 1
    return dice_sum / max(1, k)

@torch.no_grad()
def average_dice_score(pred: torch.Tensor, target: torch.Tensor, classes: int, ignore_index: int) -> float:
    mask_valid = (target != ignore_index)
    pred = pred.clone(); target = target.clone()
    pred[~mask_valid] = 0; target[~mask_valid] = 0
    dices = []
    for c in range(1, classes):
        inter = ((pred==c) & (target==c)).sum().item() * 2
        denom = (pred==c).sum().item() + (target==c).sum().item() + 1e-6
        dices.append(inter/denom)
    return float(np.mean(dices)) if dices else 0.0

# 优化器
opt_type = CONFIG["optimizer"]["type"].lower()
if opt_type == "adamw":
    optimizer = torch.optim.AdamW(model.parameters(), **CONFIG["optimizer"]["args"])
elif opt_type == "sgd":
    optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, nesterov=True, **CONFIG["optimizer"]["args"])
else:
    raise ValueError(f"未支持的优化器: {CONFIG['optimizer']['type']}")

# PolyLR（每 iter 更新）—— iters_per_epoch 按 “批次数”
poly = Poly(optimizer, epochs=CONFIG["trainer"]["epochs"], iters_per_epoch=max(1, len(train_loader)),
            power=CONFIG["lr_scheduler"]["args"].get("power", 0.9))

print("Model on:", DEVICE, "| params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


[WARN] 覆盖 class_num: config=6 -> dataset=4
[INFO] 输入通道 IN_CH = 3，类别数 class_num = 4
Model on: cpu | params: 26355364


In [18]:
# ================== Cell 7 · 训练（仅训练+验证；监控 max average_dice_score；保存 best） ==================
from tqdm.auto import tqdm, trange
from pathlib import Path

EPOCHS = CONFIG["trainer"]["epochs"]
VAL_EVERY = max(1, int(CONFIG["trainer"].get("val_per_epochs", 1)))
SAVE_PERIOD = int(CONFIG["trainer"].get("save_period", 1))
MONITOR = CONFIG["trainer"].get("monitor", "off")  # e.g. "max average_dice_score"
monitor_mode, monitor_metric = ("off", None) if MONITOR=="off" else MONITOR.split()
assert monitor_mode in ("min","max")

save_root = Path(CONFIG["trainer"]["save_dir"])
save_root.mkdir(parents=True, exist_ok=True)
best_path = save_root / "best_model.pth"

best = -np.inf if monitor_mode=="max" else np.inf
no_improve = 0
early_stop = int(CONFIG["trainer"].get("early_stop", 10))

for ep in trange(1, EPOCHS+1, desc="Epoch", unit="epoch", position=0):
    # ---- Train ----
    model.train(); run_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Train {ep}/{EPOCHS}", unit="batch", leave=False, position=1)
    for batch in pbar:
        # 兼容数据集 __getitem__ 返回 (x,y,...) 或 (x,y)
        if isinstance(batch, (list, tuple)) and len(batch) >= 2:
            x, y = batch[0], batch[1]
        else:
            raise RuntimeError("Dataset 返回格式不含标签 y。")
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = ce(logits, y) + dice_loss(logits, y, CONFIG["class_num"], CONFIG["ignore_index"])
        loss.backward(); optimizer.step(); poly.step()
        run_loss += float(loss.item())
        pbar.set_postfix(loss=f"{run_loss/(pbar.n+1):.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")
    tr_loss = run_loss / max(1, len(train_loader))

    logs = {"train_loss": tr_loss}

    # ---- Val ----
    if (val_loader is not None) and (ep % VAL_EVERY == 0):
        model.eval(); dices = []
        vbar = tqdm(val_loader, desc=f"Val   {ep}/{EPOCHS}", unit="batch", leave=False, position=2)
        with torch.no_grad():
            for batch in vbar:
                if isinstance(batch, (list, tuple)) and len(batch) >= 2:
                    x, y = batch[0], batch[1]
                else:
                    raise RuntimeError("Dataset 返回格式不含标签 y。")
                x, y = x.to(DEVICE), y.to(DEVICE)
                pred = model(x).argmax(1)
                d = average_dice_score(pred, y, CONFIG["class_num"], CONFIG["ignore_index"])
                dices.append(d); vbar.set_postfix(avg_dice=f"{np.mean(dices):.4f}")
        avg_dice = float(np.mean(dices)) if dices else 0.0
        logs["average_dice_score"] = avg_dice
        tqdm.write(f"[{ep}/{EPOCHS}] train_loss={tr_loss:.4f}  val_dice={avg_dice:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")
    else:
        tqdm.write(f"[{ep}/{EPOCHS}] train_loss={tr_loss:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}")

    # ---- periodic save（你的 save_period=100，训练 6 轮不会触发；保留语义）----
    if SAVE_PERIOD and (ep % SAVE_PERIOD == 0):
        ckpt = save_root / f"checkpoint-epoch{ep}.pth"
        torch.save({"state_dict": model.state_dict(), "epoch": ep, "config": CONFIG}, ckpt)
        tqdm.write(f"  ↳ 已保存周期权重: {ckpt}")

    # ---- monitor best ----
    if monitor_metric and monitor_metric in logs:
        cur = logs[monitor_metric]
        better = (cur > best) if monitor_mode=="max" else (cur < best)
        if better:
            best = cur; no_improve = 0
            torch.save({"state_dict": model.state_dict(), "epoch": ep, "config": CONFIG}, best_path)
            tqdm.write(f"  ↳ 已保存最优权重: {best_path}  ({monitor_mode} {monitor_metric}={best:.4f})")
        else:
            no_improve += 1
            if no_improve > early_stop:
                tqdm.write(f"早停：连续 {no_improve} 轮未提升。")
                break

tqdm.write(f"训练完成。Best {monitor_mode} {monitor_metric}: {best:.4f} | best权重: {best_path}")

D:\anaconda\envs\amdsd\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Epoch:   0%|                                                                                  | 0/6 [00:01<?, ?epoch/s]


RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 1073741824 bytes.

In [ ]:
# ================== （可选）Cell 8 · 验证集可视化导出 ==================
from PIL import Image
import cv2

# 加载最优权重
ckpt = torch.load(best_path, map_location="cpu")
eval_model = UNet(in_channels=1, num_classes=CONFIG["class_num"]).to(DEVICE)
eval_model.load_state_dict(ckpt["state_dict"])
eval_model.eval()

# 可视化目录
vis_root = Path(OUT_DIR) / "val_vis"
(vis_root/"idmask").mkdir(parents=True, exist_ok=True)
(vis_root/"overlay").mkdir(parents=True, exist_ok=True)

palette = {0:(0,0,0), 1:(255,0,0), 2:(0,0,255), 3:(0,255,0), 4:(255,255,0), 5:(255,0,255)}

@torch.no_grad()
def colorize(pred):
    rgb = np.zeros((*pred.shape,3), np.uint8)
    for k,c in palette.items(): rgb[pred==k]=c
    return rgb

# 导出一个 batch 的验证集可视化（如需全部，删除最后的 break）
for batch in val_loader:
    if isinstance(batch, (list, tuple)) and len(batch) >= 2:
        x, y, *rest = batch
    else:
        raise RuntimeError("Dataset 返回格式不含标签 y。")
    x = x.to(DEVICE)
    logits = eval_model(x)
    pred = logits.argmax(1).detach().cpu().numpy()
    for i in range(pred.shape[0]):
        pid = f"val_{i:06d}"
        pmask = pred[i].astype(np.uint8)
        Image.fromarray(pmask, mode="L").save(str(vis_root/"idmask"/f"{pid}_id.png"))
        base = (x[i,0].detach().cpu().numpy()*255).astype(np.uint8)
        base = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
        rgb  = colorize(pmask)
        overlay = cv2.addWeighted(base, 1.0, rgb, 0.45, 0)
        Image.fromarray(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)).save(str(vis_root/"overlay"/f"{pid}_ov.png"))
    break

print("验证集可视化样例已导出：", vis_root)